# 07 — Knowledge Distillation

Distillation compresses a large teacher model into a smaller student without retraining from scratch.

## Response, Feature, and Relation Distillation

**Knowledge distillation** (Hinton et al., 2015) trains a student model using the teacher's soft output distribution (logits) rather than hard labels:
$$\mathcal{L}_{KD} = \alpha \cdot T^2 \cdot KL(\sigma(z_s/T) || \sigma(z_t/T)) + (1-\alpha) \cdot \mathcal{L}_{CE}$$

The temperature *T* softens the teacher's distribution, revealing inter-class relationships (e.g., the teacher knows a '3' looks more like an '8' than a '7'). The *T²* scaling corrects for the reduced magnitude.

**Three levels of distillation**:
1. *Response distillation*: match final output logits (cheapest, most common)
2. *Feature distillation*: match intermediate layer activations — requires a projection layer if student/teacher have different widths
3. *Relation distillation*: match pairwise relationships between samples in the batch (Gram matrix matching, attention pattern matching)

**DistilBERT** (Sanh et al., 2019) uses all three: response distillation on MLM logits, feature distillation on hidden states (with linear projection), and attention map matching. Result: 40% fewer parameters, 60% faster, retaining 97% of BERT performance on GLUE.

In [ ]:
# DistilBERT-style distillation pipeline
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_classification

torch.manual_seed(42)

# Simulate teacher and student networks
class TeacherNet(nn.Module):
    def __init__(self, d=64, n_classes=5):
        super().__init__()
        self.l1 = nn.Linear(20, d)
        self.l2 = nn.Linear(d, d)
        self.l3 = nn.Linear(d, d)
        self.head = nn.Linear(d, n_classes)

    def forward(self, x):
        h1 = torch.relu(self.l1(x))
        h2 = torch.relu(self.l2(h1))
        h3 = torch.relu(self.l3(h2))
        return self.head(h3), [h1, h2, h3]


class StudentNet(nn.Module):
    def __init__(self, d=32, n_classes=5):
        super().__init__()
        self.l1 = nn.Linear(20, d)
        self.l2 = nn.Linear(d, d)
        self.head = nn.Linear(d, n_classes)

    def forward(self, x):
        h1 = torch.relu(self.l1(x))
        h2 = torch.relu(self.l2(h1))
        return self.head(h2), [h1, h2]


# Data
X, y = make_classification(n_samples=2000, n_features=20, n_classes=5,
                            n_informative=10, n_redundant=2, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y)

# Train teacher
teacher = TeacherNet(d=64)
opt_t = torch.optim.Adam(teacher.parameters(), lr=1e-3)
for _ in range(300):
    logits, _ = teacher(X)
    loss = nn.CrossEntropyLoss()(logits, y)
    opt_t.zero_grad(); loss.backward(); opt_t.step()

teacher.eval()
with torch.no_grad():
    t_acc = (teacher(X)[0].argmax(1) == y).float().mean()
print(f'Teacher accuracy: {t_acc:.3f}')
print(f'Teacher params: {sum(p.numel() for p in teacher.parameters()):,}')

In [ ]:
# Distillation training
def distillation_loss(s_logits, t_logits, labels, T=3.0, alpha=0.7):
    """Combined hard + soft distillation loss."""
    # Hard label loss
    ce_loss = F.cross_entropy(s_logits, labels)

    # Soft label loss (response distillation)
    s_soft = F.log_softmax(s_logits / T, dim=-1)
    t_soft = F.softmax(t_logits / T, dim=-1)
    kd_loss = F.kl_div(s_soft, t_soft, reduction='batchmean') * T**2

    return alpha * kd_loss + (1 - alpha) * ce_loss, ce_loss.item(), kd_loss.item()


def feature_distillation_loss(s_feats, t_feats, projectors):
    """Match intermediate layer activations."""
    # Project student features to teacher dimension
    total = 0
    for s_feat, t_feat, proj in zip(s_feats, t_feats[:len(s_feats)], projectors):
        s_proj = proj(s_feat)
        total += F.mse_loss(s_proj, t_feat.detach())
    return total


student = StudentNet(d=32)

# Feature projection layers (student dim -> teacher dim)
projectors = nn.ModuleList([
    nn.Linear(32, 64),  # s_h1 -> t_h1
    nn.Linear(32, 64),  # s_h2 -> t_h2
])

all_params = list(student.parameters()) + list(projectors.parameters())
opt_s = torch.optim.Adam(all_params, lr=1e-3)

accs = []
for step in range(500):
    idx = torch.randint(0, len(X), (128,))
    x_batch, y_batch = X[idx], y[idx]

    with torch.no_grad():
        t_logits, t_feats = teacher(x_batch)

    s_logits, s_feats = student(x_batch)
    loss_kd, ce, kd = distillation_loss(s_logits, t_logits, y_batch)
    loss_feat = feature_distillation_loss(s_feats, t_feats, projectors)
    loss = loss_kd + 0.1 * loss_feat

    opt_s.zero_grad(); loss.backward(); opt_s.step()

    if step % 100 == 0:
        with torch.no_grad():
            s_acc = (student(X)[0].argmax(1) == y).float().mean().item()
        accs.append(s_acc)
        print(f'Step {step}: acc={s_acc:.3f}, CE={ce:.4f}, KD={kd:.4f}')

student.eval()
with torch.no_grad():
    s_acc = (student(X)[0].argmax(1) == y).float().mean()
print(f'\nFinal student accuracy: {s_acc:.3f}')
print(f'Teacher accuracy: {t_acc:.3f}')
print(f'Student params: {sum(p.numel() for p in student.parameters()):,}')
print(f'Compression: {sum(p.numel() for p in teacher.parameters())/sum(p.numel() for p in student.parameters()):.1f}x')